# TradeFlow AI — nb4_eval (Final Evaluation)

**Fix #4**: Menambah evaluasi field `namaKapal`, `voyageNumber`, dan `insw_flag` sesuai Ground Truth v5.2.
Menghitung skor ANLS per field untuk memastikan NFR-007 (Akurasi >= 85%) tercapai.


In [ ]:
!pip install -q rapidfuzz transformers peft accelerate pillow bitsandbytes


In [ ]:
import os, json
from pathlib import Path
import torch
from rapidfuzz import fuzz
import numpy as np
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

def calculate_anls(gt_val, pred_val, threshold=0.5):
    if not gt_val and not pred_val: return 1.0
    if not gt_val or not pred_val: return 0.0
    gt_str, pred_str = str(gt_val).lower().strip(), str(pred_val).lower().strip()
    ed    = 1.0 - (fuzz.ratio(gt_str, pred_str) / 100.0)
    score = 1.0 - ed
    return score if score >= threshold else 0.0

REAL_DOCS_DIR = Path('/kaggle/input/tradeflow-real-docs')
GT_PATH       = REAL_DOCS_DIR / 'TradeFlow_GroundTruth_v5.2.json'

# FIX #1 applied here too: model ID corrected
BASE_MODEL_ID  = 'allenai/olmOCR-2-7B-1025'
LORA_ADAPTER_ID = 'muhammadghiffari/olm-ocr-cipl-v1'

IS_SIMULATION = True  # Ubah ke False untuk run inferensi nyata


In [ ]:
print('Memuat Base Model & LoRA Adapter...')

if not IS_SIMULATION:
    processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
    qconfig   = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID, quantization_config=qconfig, device_map='auto'
    )
    model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_ID)
    model.eval()
    print('Model siap!')
else:
    print('[MODE SIMULASI] Model tidak dimuat — menampilkan evaluasi struktur GT.')


In [ ]:
# FIX #4: Extended field list sesuai Ground Truth v5.2
# Setiap field memiliki bobot (weight) untuk weighted ANLS score
EVAL_FIELDS = {
    'nomorBl':           3.0,  # Kunci primer CEISA
    'tglBl':             2.0,  # Format tanggal sering error
    'pelabuhan_muat':    1.5,
    'pelabuhan_bongkar': 1.5,
    'container_no':      2.0,  # ISO 6346 normalization
    'beratKotor':        2.5,  # Unit conversion MTS/KGS
    'hs_code':           3.0,  # Penyebab penolakan #1
    # FIX #4: Field baru
    'namaKapal':         1.5,  # Vessel validation (PRD §5: VesselValidationAgent)
    'voyageNumber':      1.0,  # Cross-check AIS
}

def evaluate_all():
    if not GT_PATH.exists():
        print(f'File GT tidak ditemukan: {GT_PATH}')
        return

    gt_data = json.loads(GT_PATH.read_text())
    field_scores   = {f: [] for f in EVAL_FIELDS}
    insw_correct   = []   # FIX #4: Track INSW flag detection

    print('\n=== MEMULAI EVALUASI (9 Fields + INSW Flag) ===\n')

    for doc_key, gt_doc in gt_data.items():
        print(f'--- {doc_key} ({gt_doc.get("carrier", "?")})')

        ceisa = gt_doc.get('ceisa_fields', gt_doc)
        insw_gt = gt_doc.get('insw_flag', False)

        if IS_SIMULATION:
            # Mock dengan sedikit noise (20% chance error per field)
            pred_json = dict(ceisa)
            if np.random.random() < 0.2:
                pred_json['container_no'] = str(pred_json.get('container_no', '')) + 'X'
            # Mock INSW detection: model assume no INSW unless HS code starts with '28'
            hs_pred = str(pred_json.get('hs_code', ''))
            insw_pred = hs_pred.startswith('28')  # Simple heuristic for simulation
        else:
            # REAL INFERENCE (implementasi nyata di sini)
            pred_json = {}  # Ganti dengan kode inferensi asli
            insw_pred = False

        # Evaluasi per field
        for field, weight in EVAL_FIELDS.items():
            gt_val   = ceisa.get(field) or ceisa.get('container_no_normalized') if field == 'container_no' else ceisa.get(field)
            pred_val = pred_json.get(field, '')
            score    = calculate_anls(str(gt_val or ''), str(pred_val or ''))
            field_scores[field].append(score)
            icon = '✅' if score >= 0.85 else '❌'
            print(f'  {icon} {field:20}: ANLS {score:.3f} (weight={weight:.1f})')

        # FIX #4: INSW flag evaluation
        insw_match = (insw_pred == insw_gt)
        insw_correct.append(insw_match)
        insw_icon = '✅' if insw_match else '❌'
        print(f'  {insw_icon} insw_flag          : GT={insw_gt} | Pred={insw_pred}')
        print()

    # Final report
    print('\n' + '='*50)
    print('=== HASIL AKHIR WEIGHTED ANLS SCORE ===')
    print('='*50)

    weighted_scores = []
    for field, weight in EVAL_FIELDS.items():
        scores = field_scores[field]
        if not scores: continue
        avg = np.mean(scores)
        weighted_scores.extend([avg] * int(weight * 2))
        status = '✅ LULUS' if avg >= 0.85 else '❌ GAGAL'
        print(f'{field:22} : {avg:.4f}  {status}')

    insw_accuracy = np.mean(insw_correct) if insw_correct else 0.0
    insw_status   = '✅ LULUS' if insw_accuracy >= 0.9 else '❌ GAGAL'
    print(f'{"INSW Flag Detection":22} : {insw_accuracy:.4f}  {insw_status}')

    final_anls = np.mean(weighted_scores) if weighted_scores else 0.0
    print()
    print(f'WEIGHTED ANLS RATA-RATA: {final_anls:.4f}')
    print(f'INSW Detection Accuracy: {insw_accuracy:.4f}')

    print()
    if final_anls >= 0.85:
        print('🎉 MODEL LULUS NFR-007 (Akurasi >= 85%)! Siap untuk deployment.')
    else:
        print(f'⚠️  Akurasi {final_anls:.1%} belum mencapai target 85%. Perlu training lebih lanjut.')

evaluate_all()
